# RAG on an annual report: answers with page references

## Problem
Credit analysts, auditors and investors read long annual reports to find risks, changes and numbers. It takes time, and it is easy to miss something.

## まとめ（日本語）

### 処理の流れ
1. **チャンク分割**：報告書をページごとに読み込み、適切な長さのチャンクに分ける。各チャンクにページ番号を付ける。
2. **検索（Retrieval）**：質問に関係するチャンクを上位k件取ってくる。方法は3つを比較した。
   - ベクトル検索（意味の近さ）
   - BM25（キーワードの一致）
   - ハイブリッド（2つの順位をRRFで統合）
3. **回答の生成（Generation）**：取ってきたチャンクだけを使って、LLMがページ番号付きで答える。答えがなければ「Not found」と答える。

### 評価
| 評価 | 何と何を比べるか | 判定する人 |
|---|---|---|
| ① 検索の正しさ（hit@k） | 取ってきたページ ⇔ 正解のページ | 自動（正解は人間が作成） |
| ② 根拠への忠実さ（groundedness） | LLMの回答 ⇔ 取ってきた文章 | LLM（判定役） |
| ③ 回答の正しさ（correctness） | LLMの回答 ⇔ 模範解答 | LLM（判定役、模範解答は人間が作成） |

### 役割分担
**人間**
- 評価用の質問を作る
- 正解（答えが載っているページ、模範解答）を作る
- 判定役のLLMの結果が妥当かを確かめる

**システム・LLM**
- 検索：埋め込みモデルとBM25で、関係するチャンクを取ってくる
- 回答の生成：LLMが回答を作る
- ②の判定：回答が取ってきた文章に書いてあるかを、LLMが判定する
- ③の判定：回答が模範解答と合っているかを、LLMが判定する

### 結果と気づき
- ハイブリッド検索が最も良く、13問中12問で正解のページが上位3件に入った（hit@3 = 92%）。
- 年次報告書は専門用語や数字が多いため、ベクトル検索よりBM25のほうが強かった。
- 税金の質問では、回答はPillar Two（41百万ドル）に触れていたが、増加額約3億ドルの主な理由（税率の高い国からの利益増加）が抜けていた。groundednessは1.0で、最初の判定役は「correct」と判定した。
- **根拠に忠実でも、正しい答えとは限らない。** また、判定役のLLMは甘くなりやすいので、模範解答の要素を1つずつ確認させる厳しい基準に変えた。

---

## Summary (English)

### Flow
1. **Chunking**: Load the report page by page and split it into chunks of a suitable size. Each chunk keeps its page number.
2. **Retrieval**: Get the top-k chunks for a question. I compared three methods:
   - Embeddings (similar meaning)
   - BM25 (keyword match)
   - Hybrid (combine the two rankings with RRF)
3. **Generation**: The LLM answers using only the retrieved chunks, with page references. If the answer is not there, it says "Not found".

### Evaluation
| Metric | What is compared | Who judges |
|---|---|---|
| ① Retrieval (hit@k) | Retrieved pages ⇔ correct pages | Automatic (correct pages written by a human) |
| ② Groundedness | LLM answer ⇔ retrieved text | LLM judge |
| ③ Correctness | LLM answer ⇔ expected answer | LLM judge (expected answer written by a human) |

### Who did what
**Human**
- Wrote the test questions
- Wrote the correct pages and expected answers
- Checked whether the LLM judge's results made sense

**System / LLM**
- Retrieval: found relevant chunks with embeddings and BM25
- Generation: the LLM wrote the answers
- ② Judge: the LLM checked whether each claim is in the retrieved text
- ③ Judge: the LLM checked whether the answer matches the expected answer

### Results and lessons
- Hybrid search worked best: the right page was in the top 3 for 12 of 13 questions (hit@3 = 92%).
- BM25 was stronger than embeddings, because annual reports use many specific terms and numbers.
- For the tax question, the answer mentioned Pillar Two ($41 million), but missed the main reason for the ~$300 million increase (more profit from higher-tax countries). Groundedness was 1.0, and the first judge graded it as "correct".
- **A grounded answer is not always a correct answer.** LLM judges also tend to be lenient, so I made the grading stricter: check every key fact in the expected answer.

# 検索の方法：BM25・埋め込みモデル・ハイブリッド

どれも「質問に関係する文章を探す」方法。違いは**何を手がかりに探すか**。

## 1. BM25：言葉の一致で探す
- 質問の**単語が、文章にどれだけ出てくるか**で点数をつける。AIではなく単純な計算。
- 点数の工夫
  - **珍しい単語ほど重く数える**：「the」「bank」はどこにでも出るので弱い手がかり。「Pillar Two」「CET1」は数ページにしか出ないので強い手がかり。
  - **長い文章は割り引く**：長いだけで単語が多く出る文章を、不当に高く評価しないため。
- **得意**：専門用語、固有名詞、数字、略語（例：「CET1」と書いてあるページを確実に見つける）
- **苦手**：言い換え（例：「profit fell」で聞くと、「net profit declined」と書かれたページを見つけにくい）

## 2. 埋め込みモデル：意味の近さで探す
- AIのモデル（今回はOpenAIの `text-embedding-3-small`）が、文章を**意味を表すベクトル**に変える。意味が近い文章ほど、ベクトルも近くなる。
- 質問もベクトルに変えて、一番近いチャンクを探す。
- **得意**：言い換え、ふんわりした質問（例：「profit fell」と「net profit declined」を同じ意味として見つける）
- **苦手**：細かい用語や数字の区別（例：「CET1 ratio」「Tier 1 ratio」「Total capital ratio」は意味が近く、区別しにくい）

### BM25と埋め込みモデルの比較

| | BM25 | 埋め込みモデル |
|---|---|---|
| 手がかり | 単語の一致 | 意味の近さ |
| 仕組み | 単純な計算 | AIのモデル |
| 費用 | 無料 | API代（少額） |
| 得意 | 専門用語、固有名詞、数字 | 言い換え、ふんわりした質問 |
| 苦手 | 言い換え | 細かい用語の区別 |

## 3. ハイブリッド：2つの結果を1つにまとめる
BM25と埋め込みモデルの検索結果を、**RRF（Reciprocal Rank Fusion）**で1つのランキングにまとめる。

### 手順
1. BM25で上位20件を取る。
2. 埋め込みモデルでも上位20件を取る。
3. 各チャンクに**順位をもとにした点数**をつけて足し合わせる。
   - 点数 = 1 / (60 + 順位)
   - 1位なら 1/61、2位なら 1/62。順位が下がるほど少しずつ減る。
4. 合計点の高い順に並べて、上位5件をLLMに渡す。

### 例

| チャンク | BM25の順位 | 埋め込みの順位 | 合計点 |
|---|---|---|---|
| A | 1位 | 3位 | 1/61 + 1/63 = **0.0323** |
| B | 2位 | 圏外 | 1/62 = 0.0161 |
| C | 圏外 | 1位 | 1/61 = 0.0164 |

- **両方で上位なら最優先**（A）。
- **片方でしか見つからなくても、上位なら拾う**（B、C）。
- BM25が苦手な言い換えを埋め込みが拾い、埋め込みが苦手な専門用語をBM25が拾う。

### なぜ点数ではなく「順位」で足すのか
- BM25の点数には上限がない（5や12など）。
- 埋め込みの点数（コサイン類似度）は、0.3〜0.8くらいの範囲。
- ものさしが違うので、そのまま足すとBM25が強くなりすぎる。順位に直せば同じものさしで公平に混ぜられる。

### 「60」とは
- 順位の差をどれくらい重く見るかを決める数字。RRFを提案した論文で使われた定番の値。
- 大きいほど1位と10位の点数差が小さくなり、片方の1位だけが強くなりすぎるのを防ぐ。

## 4. 今回の結果（OCBC Annual Report 2025、13問）

| 方法 | hit@3 |
|---|---|
| 埋め込みモデル | 62% |
| BM25 | 85% |
| ハイブリッド | 92% |

- 年次報告書の質問は専門用語が多いため、**BM25が強かった**。
- **ハイブリッドが最も良かった**のは、2つの方法がお互いの苦手を補い合ったため。

## 1. Setup

In [9]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel
from pypdf import PdfReader
from rank_bm25 import BM25Okapi

pd.set_option("display.max_colwidth", 120)

load_dotenv("../.env")                      

PDF_PATH = Path("../data/raw/report.pdf")
CACHE_DIR = Path("../data/processed/rag")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"                 
CHUNK_SIZE = 1200                          
CHUNK_OVERLAP = 200
TOP_K = 5

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

## 2. Load the PDF and split it into chunks

In [10]:
reader = PdfReader(PDF_PATH)
pages = [Document(page_content=" ".join((p.extract_text() or "").split()), metadata={"page": i + 1})
         for i, p in enumerate(reader.pages)]
pages = [d for d in pages if len(d.page_content) > 50]       # skip empty / image-only pages
print(f"{len(reader.pages)} pages in the PDF, {len(pages)} with text")

# split each page separately, so every chunk has one page number
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(pages)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i
print(f"{len(chunks)} chunks")
print(chunks[min(10, len(chunks) - 1)])

276 pages in the PDF, 273 with text
853 chunks
page_content='33% growth (y-o-y) in Group wealth management fees – a record high 30% growth (y-o-y) in revenue from retail bancassurance products across all our core markets 25% growth (y-o-y) in assets under management from Bank of Singapore’s ultra-high-net-worth clients >70% growth (y-o-y) in investment and insurance income in Hong Kong and Macau Wealth Management 28% increase (y-o-y) in profit contribution from Great Eastern Holdings 45% growth (y-o-y) in new business value in Singapore (Great Eastern) Insurance 1st gold fund in Singapore with physical gold bars custodised entirely in Singapore (Lion Global Investors) 1st actively managed bond ETF in Singapore (Lion Global Investors) Asset Management 5Annual Report 2025' metadata={'page': 7, 'chunk_id': 10}


## 3. Vector store (embeddings)
The vector store is saved on disk, so the report is only embedded once.

In [11]:
store_path = CACHE_DIR / f"{PDF_PATH.stem}_{EMBED_MODEL}_{CHUNK_SIZE}_{CHUNK_OVERLAP}.json"
if store_path.exists():
    vectorstore = InMemoryVectorStore.load(str(store_path), embeddings)
else:
    t0 = time.time()
    vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)
    vectorstore.dump(str(store_path))
    print(f"embedded {len(chunks)} chunks in {time.time() - t0:.0f}s")

embedded 853 chunks in 4s


## 4. Retrieval: embeddings, BM25 and hybrid
- **Embeddings** find text with a similar meaning, even with different words ("sales fell" vs "revenue decreased").
- **BM25** finds exact keywords. It is strong for names, product codes and accounting terms.
- **Hybrid**: combine the two rankings with Reciprocal Rank Fusion (RRF): score = sum of 1 / (60 + rank).

In [16]:
import re

tokenize = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = BM25Okapi([tokenize(c.page_content) for c in chunks])


def search_dense(question, k=TOP_K):
    return vectorstore.similarity_search(question, k=k)


def search_bm25(question, k=TOP_K):
    top = np.argsort(-bm25.get_scores(tokenize(question)))[:k]
    return [chunks[i] for i in top]


def search_hybrid(question, k=TOP_K, pool=20, rrf_k=60):
    score = {}
    for docs in (search_dense(question, pool), search_bm25(question, pool)):
        for rank, d in enumerate(docs, start=1):
            cid = d.metadata["chunk_id"]
            score[cid] = score.get(cid, 0) + 1 / (rrf_k + rank)
    best = sorted(score, key=score.get, reverse=True)[:k]
    return [chunks[i] for i in best]


SEARCH = {"dense": search_dense, "bm25": search_bm25, "hybrid": search_hybrid}

pd.DataFrame([{"page": d.metadata["page"], "text": d.page_content[:200]}
              for d in search_hybrid("What are the main risk factors?")])

,page,text
0,214,"for its business use. Country Risk The Group’s country risk framework covers the assessment and rating of countries,..."
1,83,"support effective risk management. Principal Risk Types Risks are categorised into five principal risk types, each m..."
2,83,"business goals, and manage potential impacts under adverse circumstances. Risks are identified, measured and monitor..."
3,89,"The design of these limits considers direct and indirect risk drivers, such as economic sector, industry and geograp..."
4,239,"adjustment methodologies, independent price testing, and identifying valuation gaps. Valuation policies are formulat..."


## 5. Answer with page references (LCEL chain)
The prompt says: use only the context, cite pages, and say "not found" when the report does not answer the question. This is the main protection against made-up answers.

In [19]:
prompt = ChatPromptTemplate.from_messages([
    # ("system",
    #  "You answer questions about an annual report. "
    #  "Use ONLY the context. After each sentence, cite the page like [p. 12]. "
    #  "If the context does not contain the answer, reply exactly: Not found in the report. "
    #  "Keep the answer short and factual."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ("system",
    "You answer questions about an annual report. "
    "Use ONLY the context. After each sentence, cite the page like [p. 12]. "
    "Answer with what the context says, even if it covers only part of the question. "
    "Reply 'Not found in the report.' only if the context has nothing related to the question. "
    "Keep the answer short and factual."),
])
answer_chain = prompt | llm | StrOutputParser()


def format_context(docs):
    return "\n\n".join(f"[p. {d.metadata['page']}] {d.page_content}" for d in docs)


def ask(question, method="hybrid", k=TOP_K):
    docs = SEARCH[method](question, k)
    context = format_context(docs)
    return {"question": question, "pages": [d.metadata["page"] for d in docs], "context": context,
            "answer": answer_chain.invoke({"context": context, "question": question})}


for q in ["What are the main risk factors for the company?",
          "Why did revenue change compared with last year?",
          "Is the company involved in any major lawsuits?",
          "Who won the football World Cup in 2022?"]:          # not in the report -> should say "Not found"
    r = ask(q)
    print(f"Q: {q}\nretrieved pages: {r['pages']}\n{r['answer']}\n" + "-" * 80)

Q: What are the main risk factors for the company?
retrieved pages: [214, 83, 239, 210, 86]
The main risk factors for the company include country risk, market risk, liquidity risk, model risk, and valuation risk. Country risk involves the assessment and rating of countries and their political and economic outlook, affecting cross-border transactions [p. 214]. Market risk arises from fluctuations in market factors such as interest rates, foreign exchange rates, and commodity prices, particularly from trading and balance sheet management activities [p. 214]. Liquidity risk is also a concern, as it relates to the company's ability to meet its financial obligations [p. 214]. Model risk pertains to limitations in the valuation models used, which can affect financial assessments [p. 239]. Valuation risk includes adjustments for uncertainties in input parameters and model deficiencies [p. 239].
--------------------------------------------------------------------------------
Q: Why did revenue

## 6. Evaluation

### 6.1 Retrieval: did we find the right page?
Write a few questions **after reading the report**, with the page where the answer is. Then check if that page is in the top-k chunks (hit rate @k).

Replace the examples below with your own (8–15 questions are enough).

In [20]:
EVAL_SET = [
    # (question, pages where the answer is, expected answer for my own check)
    ("What was the Group net profit for 2025?", [4, 6, 9, 13, 99], "$7.42 billion"),
    ("Why did net interest income fall in 2025?", [99], "asset yields fell faster than funding costs as benchmark rates dropped"),
    ("What was the net interest margin in 2025?", [98, 101], "1.91% (2.20% in 2024)"),
    ("What drove the growth in non-interest income?", [99], "fee, trading and insurance income; wealth fees +33%"),
    ("What was the CET1 capital ratio at the end of 2025?", [4, 25, 78, 98, 100], "16.9% (15.1% fully phased-in)"),
    ("What was the non-performing loan ratio?", [25, 38, 98, 99, 100], "0.9%"),
    ("What were the credit costs in 2025?", [9, 25, 38, 99, 100], "17 bp, down from 19 bp"),
    ("What is the total dividend per share for 2025?", [10, 77, 99, 100], "99 cents (83 ordinary + 16 special)"),
    ("What is the capital return plan?", [5, 10, 25, 77, 99, 100], "$2.5 billion over two years, done by FY2026"),
    ("Why did tax expense increase?", [99], "more profit from higher-tax places, and BEPS Pillar Two"),
    ("How is the bank using agentic AI?", [20, 21, 49], "in KYC: Bank of Singapore's Source of Wealth Assistant"),
    ("Which sector needed extra attention for credit risk?", [9, 38], "commercial real estate in some markets"),
    ("How does the bank detect suspicious networks for money laundering?", [94], "risk surveillance with AI and data analytics"),
]


def hit_rate(method, ks=(1, 3, 5)):
    ranks = []
    for q, true_pages, _ in EVAL_SET:
        found = [d.metadata["page"] for d in SEARCH[method](q, max(ks))]
        hit = [i for i, p in enumerate(found) if p in true_pages]
        ranks.append(hit[0] + 1 if hit else np.inf)
    return {f"hit@{k}": np.mean(np.array(ranks) <= k) for k in ks}


pd.DataFrame({m: hit_rate(m) for m in SEARCH}).T

,hit@1,hit@3,hit@5
dense,0.461538,0.615385,0.692308
bm25,0.538462,0.846154,0.846154
hybrid,0.538462,0.923077,0.923077


### 6.2 Answer: is it supported by the text? (groundedness)
A second LLM call reads the context and the answer, and checks each claim (LLM-as-a-judge). `with_structured_output` makes the LLM return a fixed format. The judge is not perfect, so I also read some answers myself.

In [21]:
class Claim(BaseModel):
    claim: str
    supported: bool


class Judgement(BaseModel):
    claims: list[Claim]


judge_prompt = ChatPromptTemplate.from_messages([
    ("system", "You check an answer against its context. Split the answer into short claims. "
               "For each claim, say if the context supports it."),
    ("human", "Context:\n{context}\n\nAnswer:\n{answer}"),
])
judge_chain = judge_prompt | llm.with_structured_output(Judgement)


def groundedness(r):
    if r["answer"].startswith("Not found"):
        return np.nan
    claims = judge_chain.invoke({"context": r["context"], "answer": r["answer"]}).claims
    return np.mean([c.supported for c in claims]) if claims else np.nan


results = []
for q, true_pages, expected in EVAL_SET:
    r = ask(q)
    results.append({"question": q, "expected": expected,
                    "page retrieved": any(p in true_pages for p in r["pages"]),
                    "groundedness": groundedness(r), "answer": r["answer"]})
pd.DataFrame(results)

,question,expected,page retrieved,groundedness,answer
0,What was the Group net profit for 2025?,$7.42 billion,True,1.0,"The Group net profit for 2025 was $7.42 billion, which was 2% below the record $7.59 billion in the previous year [p..."
1,Why did net interest income fall in 2025?,asset yields fell faster than funding costs as benchmark rates dropped,True,1.0,Net interest income fell in 2025 primarily due to asset yields declining faster than funding costs as key benchmark ...
2,What was the net interest margin in 2025?,1.91% (2.20% in 2024),True,1.0,The net interest margin in 2025 was 1.91% [p. 101].
3,What drove the growth in non-interest income?,"fee, trading and insurance income; wealth fees +33%",True,1.0,"The growth in non-interest income was driven by double-digit growth across fee, trading, and insurance income, which..."
4,What was the CET1 capital ratio at the end of 2025?,16.9% (15.1% fully phased-in),True,1.0,The CET1 capital ratio at the end of 2025 was 16.9% on a transitional basis and 15.1% on a fully phased-in basis [p....
5,What was the non-performing loan ratio?,0.9%,True,1.0,The non-performing loan (NPL) ratio was stable at 0.9% for 2025 [p. 99].
6,What were the credit costs in 2025?,"17 bp, down from 19 bp",True,1.0,"The credit costs in 2025 were lower at 17 basis points of loans, compared to 19 basis points a year ago [p. 99]."
7,What is the total dividend per share for 2025?,99 cents (83 ordinary + 16 special),True,1.0,The total dividend per share for 2025 is 99 cents [p. 10].
8,What is the capital return plan?,"$2.5 billion over two years, done by FY2026",True,1.0,The capital return plan is a comprehensive approach to return $2.5 billion of capital to shareholders over two years...
9,Why did tax expense increase?,"more profit from higher-tax places, and BEPS Pillar Two",False,1.0,"The tax expense increased due to a rise in current tax expense, which was $1,486 million in 2025 compared to $1,178 ..."


### 6.3 Correctness: does the answer match the expected answer?
Groundedness only checks the answer against the retrieved text. Here, another LLM call compares the answer with the **expected answer** I wrote from the report.

In [22]:
from typing import Literal


class Correctness(BaseModel):
    verdict: Literal["correct", "partial", "incorrect"]
    reason: str


correct_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You grade an answer to a question about an annual report, by comparing it with the expected answer. "
     "'correct': it gives the key facts of the expected answer. "
     "'partial': some key facts are missing. "
     "'incorrect': it is wrong, or it does not answer the question. "
     "Ignore wording and extra details. Give a short reason."),
    ("human", "Question: {question}\n\nExpected answer: {expected}\n\nAnswer: {answer}"),
])
correct_chain = correct_prompt | llm.with_structured_output(Correctness)

df = pd.DataFrame(results)
graded = [correct_chain.invoke({"question": r.question, "expected": r.expected, "answer": r.answer})
          for r in df.itertuples()]
df["correctness"] = [g.verdict for g in graded]
df["reason"] = [g.reason for g in graded]

print(df["correctness"].value_counts())
df[["question", "page retrieved", "groundedness", "correctness", "reason"]]

correctness
correct    13
Name: count, dtype: int64


,question,page retrieved,groundedness,correctness,reason
0,What was the Group net profit for 2025?,True,1.0,correct,The answer provides the exact figure for the Group net profit for 2025 as stated in the expected answer.
1,Why did net interest income fall in 2025?,True,1.0,correct,"The answer provides the key fact that asset yields fell faster than funding costs as benchmark rates dropped, which ..."
2,What was the net interest margin in 2025?,True,1.0,correct,The response provides the exact net interest margin for 2025 as stated in the expected answer.
3,What drove the growth in non-interest income?,True,1.0,correct,"The answer includes all key facts from the expected answer, detailing the sources of growth in non-interest income a..."
4,What was the CET1 capital ratio at the end of 2025?,True,1.0,correct,"The answer provides both key facts: the CET1 capital ratio of 16.9% and the fully phased-in ratio of 15.1%, matching..."
5,What was the non-performing loan ratio?,True,1.0,correct,"The answer provides the key fact of the expected answer, stating the non-performing loan ratio as 0.9%."
6,What were the credit costs in 2025?,True,1.0,correct,The response accurately states the credit costs in 2025 as 17 bp and compares it to the previous year's 19 bp.
7,What is the total dividend per share for 2025?,True,1.0,correct,The answer provides the total dividend per share for 2025 and matches the expected answer.
8,What is the capital return plan?,True,1.0,correct,"The answer provides the key facts of the expected answer, including the total amount ($2.5 billion), the timeframe (..."
9,Why did tax expense increase?,False,1.0,correct,"The answer provides key facts about the increase in tax expense, including the rise in current tax expense and the r..."


In [23]:
print(df.loc[9, "answer"])
print()
print(df.loc[9, "reason"])

The tax expense increased due to a rise in current tax expense, which was $1,486 million in 2025 compared to $1,178 million in 2024 [p. 161]. Additionally, the Group recognized a Pillar Two current income tax expense of $41 million for the year, which is included in the income tax expense [p. 162].

The answer provides key facts about the increase in tax expense, including the rise in current tax expense and the recognition of the Pillar Two current income tax expense.


## 7. Notes
Write after running:
- Which retrieval method found the right page most often? Did hybrid help for questions with specific terms?
- Were there answers with unsupported claims? What kind (numbers, tables)?
- Did the model say "Not found" for the question that is not in the report?

Known limits: tables in PDFs often lose their layout when extracted, so numbers from tables are the weakest point. Next steps could be table-aware parsing, and re-ranking the retrieved chunks.

まとめ


フロー
- テキストをチャンクにわける（適切なサイズでわける）
- RAGで必要な箇所をとってくる：3つの方法（ベクトル、BM25、hybrid）→ 上位k件を指定、何％正しくとってきたかを評価（人間の答え）
- とってきた箇所から質問の答えを生成させる：①とってきたページが正しいか（人間の答えとの一致） ②答えがRagと一致しているか（生成ＡＩが判定）③答えが模範解答と比べて正しいか（人間が答え、生成ＡＩで一致を判定）


人間がやったこと
- 問題を作る
- 問題の答えを作る（ページ数、回答）


生成AIがやったこと
- Ragをとってくる
- 生成ＡＩで問題の回答を作成する
- Ragと生成ＡＩ回答が一致しているか
- 生成ＡＩ回答と模範回答が一致しているか

